In [2]:
%pip install --quiet pandas pyarrow geopandas shapely matplotlib openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Pipeline ETL — Setor Censitário × CEP × Renda (São Paulo)

Pipeline consolidada **em um único notebook**, do dado bruto IBGE até o parquet final com renda imputada.

**Entradas (todas do IBGE, baixadas separadamente):**
- CNEFE 2022 (`saida_cnefe_uf/extraido/SEM_UF/35_SP.csv`) — endereços com lat/lng
- Shapefile CD2022 (`BR_setores_CD2022/`)
- Agregados de renda IBGE (`Agregados_por_setores_renda_responsavel_BR.xlsx`)

**Saída final em `saida_etl_final_sp/`:**
- `renda_setor_cep_sp_final.parquet`
- `renda_setor_cep_sp_final.csv`
- `renda_setor_cep_sp_final_resumo.json`

**Resultado esperado (SP):** 103.319 setores no shapefile → output com renda+CEP em ~97,25% (100.475), com flag `origem_renda` distinguindo `original`, `imputado_*` e `sem_dado_legitimo_*`.

**Pipeline em 8 etapas:**

1. Carregar renda XLSX (driver)
2. Geofencing CNEFE → setor CD2022 (uma passada, escrevendo dupla atribuição: geométrica + COD_SETOR original)
3. Agregar setor → CEPs (cascata `setor_cep_geo` → `setor_cep_orig`)
4. Atributos do shapefile (com CD_BAIRRO/CD_DIST/CD_SUBDIST pra imputação)
5. Join driver=renda (outer + categoria `fora_csv_ibge`)
6. Estatísticas intermediárias
7. Imputação de renda em cascata (bairro → subdistrito → distrito → município) só pra `CD_TIPO ∈ {0,1}`
8. Estatísticas finais + exportar

## Setup — imports, paths e parâmetros

In [3]:
from pathlib import Path
import json
import sqlite3
import time
from datetime import datetime, timezone

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

import geopandas as gpd
from shapely.geometry import Point

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')


def log(msg):
    print(msg, flush=True)


def now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

In [4]:
def find_project_root(start):
    start = Path(start).resolve()
    for c in [start, *start.parents]:
        if (c / 'saida_cnefe_uf').exists() and (c / 'BR_setores_CD2022').exists():
            return c
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
CNEFE_ROOT   = PROJECT_ROOT / 'saida_cnefe_uf' / 'extraido' / 'SEM_UF'
RENDA_FILE   = PROJECT_ROOT / 'Agregados_por_setores_renda_responsavel_BR_csv' / 'Agregados_por_setores_renda_responsavel_BR.xlsx'
SHAPEFILE    = PROJECT_ROOT / 'BR_setores_CD2022' / 'BR_setores_CD2022.shp'

OUTPUT_DIR   = PROJECT_ROOT / 'saida_etl_final_sp'
WORK_SQLITE  = OUTPUT_DIR / 'etl_final_sp_work.sqlite'
OUT_PARQUET  = OUTPUT_DIR / 'renda_setor_cep_sp_final.parquet'
OUT_CSV      = OUTPUT_DIR / 'renda_setor_cep_sp_final.csv'
OUT_RESUMO   = OUTPUT_DIR / 'renda_setor_cep_sp_final_resumo.json'

# Filtro: pipeline desenhada para SP.
UF_CODIGO = '35'
UF_SIGLA  = 'SP'

# Parametros performance.
CHUNKSIZE = 250_000

# Reconstruir SQLite do zero ou reaproveitar?
REBUILD_SQLITE = True

# Imputacao.
TIPOS_IMPUTAVEIS = {'0', '1'}   # CD_TIPO 0=urbano comum, 1=rural comum
MIN_VIZINHOS = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('CNEFE_ROOT   :', CNEFE_ROOT)
print('RENDA_FILE   :', RENDA_FILE)
print('SHAPEFILE    :', SHAPEFILE)
print('OUTPUT_DIR   :', OUTPUT_DIR)
print('UF           :', UF_SIGLA, '(', UF_CODIGO, ')')

PROJECT_ROOT : C:\Users\matpa\Documents\Base CEP_LOG_RENDA
CNEFE_ROOT   : C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_cnefe_uf\extraido\SEM_UF
RENDA_FILE   : C:\Users\matpa\Documents\Base CEP_LOG_RENDA\Agregados_por_setores_renda_responsavel_BR_csv\Agregados_por_setores_renda_responsavel_BR.xlsx
SHAPEFILE    : C:\Users\matpa\Documents\Base CEP_LOG_RENDA\BR_setores_CD2022\BR_setores_CD2022.shp
OUTPUT_DIR   : C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp
UF           : SP ( 35 )


In [5]:
SIGLA_POR_UF = {
    '11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
    '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL',
    '28':'SE','29':'BA','31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR',
    '42':'SC','43':'RS','50':'MS','51':'MT','52':'GO','53':'DF',
}

REQUIRED_CNEFE_COLUMNS = ['CEP', 'COD_SETOR', 'LATITUDE', 'LONGITUDE', 'NV_GEO_COORD']
REQUIRED_RENDA_COLUMNS = ['CD_SETOR', 'V06001', 'V06002', 'V06003', 'V06004', 'V06005', 'V06006']


def parse_br_number(series):
    # XLSX do IBGE armazena numeros como float en-US (ponto decimal).
    # pd.to_numeric converte direto e devolve NaN para 'X' (sigilo).
    return pd.to_numeric(series, errors='coerce')


def open_sqlite(p):
    p.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(p)
    conn.execute('PRAGMA journal_mode = WAL')
    conn.execute('PRAGMA synchronous = NORMAL')
    conn.execute('PRAGMA temp_store = MEMORY')
    conn.execute('PRAGMA cache_size = -200000')
    conn.execute('PRAGMA mmap_size = 30000000000')
    return conn


def reset_tables(conn):
    conn.executescript('''
        DROP TABLE IF EXISTS setor_cep_geo;
        DROP TABLE IF EXISTS setor_cep_orig;
        CREATE TABLE setor_cep_geo (
            cd_setor TEXT NOT NULL,
            cep TEXT NOT NULL,
            qtd_enderecos INTEGER NOT NULL,
            PRIMARY KEY (cd_setor, cep)
        ) WITHOUT ROWID;
        CREATE TABLE setor_cep_orig (
            cd_setor TEXT NOT NULL,
            cep TEXT NOT NULL,
            qtd_enderecos INTEGER NOT NULL,
            PRIMARY KEY (cd_setor, cep)
        ) WITHOUT ROWID;
    ''')
    conn.commit()

## Etapa 1 — Renda como driver (XLSX)

Lê o XLSX com renda média (V06004) e renda mediana (V06006). Classifica `motivo_renda` distinguindo `numerica`, `sigilo` (X) e `ausente`.

In [6]:
def load_renda(path, uf_codigo):
    log(f'[renda] Lendo {path}')
    df = pd.read_excel(path, dtype=str, keep_default_na=False, na_filter=False)
    missing = [c for c in REQUIRED_RENDA_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f'Colunas ausentes: {missing}')
    df['cd_setor'] = df['CD_SETOR'].astype(str).str.strip()
    df = df.loc[df['cd_setor'].str.len() == 15].copy()
    df = df.loc[df['cd_setor'].str.slice(0, 2) == uf_codigo].copy()

    vars_renda = ['V06001','V06002','V06003','V06004','V06005','V06006']
    for v in vars_renda:
        df[f'raw_{v.lower()}'] = df[v].astype(str).str.strip()
        df[f'renda_{v.lower()}'] = parse_br_number(df[v])

    def classify(row):
        if pd.notna(row['renda_v06004']):
            return 'numerica'
        if row['raw_v06004'].upper() == 'X':
            return 'sigilo'
        return 'ausente'

    df['motivo_renda'] = df.apply(classify, axis=1)
    keep = ['cd_setor', 'motivo_renda'] + [f'renda_{v.lower()}' for v in vars_renda]
    out = df[keep].drop_duplicates('cd_setor', keep='last').reset_index(drop=True)
    log(f'[renda] Setores no driver: {len(out):,}')
    print(out['motivo_renda'].value_counts())
    return out


renda_df = load_renda(RENDA_FILE, UF_CODIGO)
display(renda_df.head())

[renda] Lendo C:\Users\matpa\Documents\Base CEP_LOG_RENDA\Agregados_por_setores_renda_responsavel_BR_csv\Agregados_por_setores_renda_responsavel_BR.xlsx
[renda] Setores no driver: 100,928
motivo_renda
numerica    99223
sigilo       1697
ausente         8
Name: count, dtype: int64


,cd_setor,motivo_renda,renda_v06001,renda_v06002,renda_v06003,renda_v06004,renda_v06005,renda_v06006
0,350010505000001,numerica,138.0000,288.0000,1.4400,"4,107.2900","24,164,514.2500","2,500.0000"
1,350010505000002,numerica,280.0000,674.0000,2.1500,"4,446.5500","23,063,356.4800","3,000.0000"
2,350010505000003,numerica,198.0000,379.0000,1.0400,"4,713.6800","19,642,814.0700","3,050.0000"
3,350010505000004,numerica,236.0000,566.0000,1.4700,"3,544.1600","18,932,163.4400","2,001.0000"
4,350010505000005,numerica,272.0000,604.0000,1.2400,"4,448.4400","20,210,229.8700","3,000.0000"


## Etapa 2 — Geofencing (passada única do CNEFE)

Carrega geometria CD2022 do SP e processa o arquivo CNEFE `35_SP.csv` em chunks de 250k linhas. Para cada chunk:

1. Constrói GeoDataFrame de pontos `(LATITUDE, LONGITUDE)`.
2. `gpd.sjoin(...predicate='within')` → atribui `cd_setor` pela geometria.
3. Grava **simultaneamente** em duas tabelas SQLite:
   - `setor_cep_geo` — com o `cd_setor` derivado pela geometria (primário).
   - `setor_cep_orig` — com o `COD_SETOR` original do CNEFE (fallback para micro-setores em borda).

Essa dupla escrita é o que sustenta a estratégia híbrida na Etapa 3.

In [7]:
def load_shapefile_geometry(shapefile, uf_codigo):
    log(f'[shapefile] Carregando geometria de {shapefile}')
    gdf = gpd.read_file(shapefile, columns=['CD_SETOR', 'CD_UF', 'geometry'])
    gdf['cd_setor'] = gdf['CD_SETOR'].astype(str).str.strip()
    gdf['cod_uf']   = gdf['CD_UF'].astype(str).str.strip().str.zfill(2)
    log(f'[shapefile] Total Brasil: {len(gdf):,}  |  CRS: {gdf.crs}')
    gdf = gdf.loc[gdf['cod_uf'] == uf_codigo]
    gdf = gdf.loc[gdf['cd_setor'].str.len() == 15, ['cd_setor', 'geometry']].reset_index(drop=True)
    log(f'[shapefile] Setores em {uf_codigo}: {len(gdf):,}')
    if len(gdf) == 0:
        raise RuntimeError('Filtro UF zerou — verifique parametros.')
    return gdf


shp_geo = load_shapefile_geometry(SHAPEFILE, UF_CODIGO)
display(shp_geo.head(3))

[shapefile] Carregando geometria de C:\Users\matpa\Documents\Base CEP_LOG_RENDA\BR_setores_CD2022\BR_setores_CD2022.shp
[shapefile] Total Brasil: 468,099  |  CRS: EPSG:4674
[shapefile] Setores em 35: 103,319


,cd_setor,geometry
0,351300905000799,"POLYGON ((-46.88061 -23.60319, -46.8808 -23.60..."
1,351510305000201,"POLYGON ((-46.83274 -23.88221, -46.83216 -23.8..."
2,352940105000533,"POLYGON ((-46.46543 -23.67786, -46.46549 -23.6..."


In [8]:
def ingest_cnefe(conn, csv_path, shp_geo, chunksize):
    target_crs = shp_geo.crs
    stats = {'rows_read': 0, 'via_geo': 0, 'via_orig_only': 0, 'invalidos': 0}
    log(f'[cnefe] Processando {csv_path.name}')
    t0 = time.time()
    reader = pd.read_csv(
        csv_path, sep=';', dtype=str,
        usecols=REQUIRED_CNEFE_COLUMNS,
        keep_default_na=False, na_filter=False, chunksize=chunksize,
    )
    chunk_i = 0
    for chunk in reader:
        chunk_i += 1
        raw = len(chunk)
        if raw == 0:
            continue
        lat = pd.to_numeric(chunk['LATITUDE'].str.replace(',', '.', regex=False), errors='coerce')
        lng = pd.to_numeric(chunk['LONGITUDE'].str.replace(',', '.', regex=False), errors='coerce')
        cep = chunk['CEP'].fillna('').astype(str).str.replace(r'\D+', '', regex=True)
        cep = cep.where(cep.eq(''), cep.str.zfill(8))
        cnefe_setor = chunk['COD_SETOR'].fillna('').astype(str).str.strip().str.slice(0, 15)
        mask = lat.notna() & lng.notna() & cep.ne('') & cnefe_setor.str.len().eq(15)
        invalid = int((~mask).sum())
        if not mask.any():
            stats['rows_read'] += raw
            stats['invalidos'] += invalid
            continue

        # Grava setor_cep_orig (sempre, com cd_setor do CNEFE).
        orig_df = pd.DataFrame({
            'cd_setor': cnefe_setor[mask].values,
            'cep': cep[mask].values,
        })
        orig_grouped = (
            orig_df.groupby(['cd_setor', 'cep'], sort=False)
                   .size().reset_index(name='qtd_enderecos')
        )
        with conn:
            conn.executemany(
                'INSERT INTO setor_cep_orig (cd_setor, cep, qtd_enderecos) VALUES (?, ?, ?) '
                'ON CONFLICT(cd_setor, cep) DO UPDATE SET '
                'qtd_enderecos = setor_cep_orig.qtd_enderecos + excluded.qtd_enderecos',
                orig_grouped.itertuples(index=False, name=None),
            )

        # Geofencing: ponto-no-poligono contra shp_geo.
        pts = gpd.GeoDataFrame(
            {'cep': cep[mask].values},
            geometry=gpd.points_from_xy(lng[mask], lat[mask]),
            crs=target_crs,
        )
        joined = gpd.sjoin(pts, shp_geo, how='left', predicate='within')
        has_geo = joined['cd_setor'].notna()
        if has_geo.any():
            geo_df = pd.DataFrame({
                'cd_setor': joined.loc[has_geo, 'cd_setor'].values,
                'cep': joined.loc[has_geo, 'cep'].values,
            })
            geo_grouped = (
                geo_df.groupby(['cd_setor', 'cep'], sort=False)
                      .size().reset_index(name='qtd_enderecos')
            )
            with conn:
                conn.executemany(
                    'INSERT INTO setor_cep_geo (cd_setor, cep, qtd_enderecos) VALUES (?, ?, ?) '
                    'ON CONFLICT(cd_setor, cep) DO UPDATE SET '
                    'qtd_enderecos = setor_cep_geo.qtd_enderecos + excluded.qtd_enderecos',
                    geo_grouped.itertuples(index=False, name=None),
                )
        via_g = int(has_geo.sum())
        via_o = int((~has_geo).sum())
        stats['rows_read'] += raw
        stats['via_geo'] += via_g
        stats['via_orig_only'] += via_o
        stats['invalidos'] += invalid
        log(f'  chunk {chunk_i}: {raw:,} linhas | via_geo={via_g:,} | so_orig={via_o:,}')
    log(f'[cnefe] Tempo total: {(time.time()-t0)/60:.1f} min')
    return stats


cnefe_csv = CNEFE_ROOT / f'{UF_CODIGO}_{UF_SIGLA}.csv'
if not cnefe_csv.exists():
    raise FileNotFoundError(f'CNEFE nao encontrado: {cnefe_csv}')
log(f'CNEFE CSV: {cnefe_csv}  ({cnefe_csv.stat().st_size/1e9:.2f} GB)')

t0 = time.time()
with open_sqlite(WORK_SQLITE) as conn:
    if REBUILD_SQLITE:
        reset_tables(conn)
        cnefe_stats = ingest_cnefe(conn, cnefe_csv, shp_geo, CHUNKSIZE)
        conn.execute('CREATE INDEX IF NOT EXISTS idx_geo_cd  ON setor_cep_geo(cd_setor)')
        conn.execute('CREATE INDEX IF NOT EXISTS idx_orig_cd ON setor_cep_orig(cd_setor)')
        conn.commit()
    else:
        cnefe_stats = {'info': 'SQLite reaproveitado'}
    total_geo  = conn.execute('SELECT COUNT(*) FROM setor_cep_geo').fetchone()[0]
    total_orig = conn.execute('SELECT COUNT(*) FROM setor_cep_orig').fetchone()[0]
log(f'[etapa 2] Tempo: {(time.time()-t0)/60:.1f} min')
log(f'[etapa 2] Pares (setor, cep) em setor_cep_geo : {total_geo:,}')
log(f'[etapa 2] Pares (setor, cep) em setor_cep_orig: {total_orig:,}')
log(f'[etapa 2] Stats: {cnefe_stats}')

CNEFE CSV: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_cnefe_uf\extraido\SEM_UF\35_SP.csv  (3.83 GB)
[cnefe] Processando 35_SP.csv
  chunk 1: 250,000 linhas | via_geo=249,999 | so_orig=1
  chunk 2: 250,000 linhas | via_geo=249,999 | so_orig=1
  chunk 3: 250,000 linhas | via_geo=250,000 | so_orig=0
  chunk 4: 250,000 linhas | via_geo=250,000 | so_orig=0
  chunk 5: 250,000 linhas | via_geo=250,000 | so_orig=0
  chunk 6: 250,000 linhas | via_geo=249,993 | so_orig=7
  chunk 7: 250,000 linhas | via_geo=249,995 | so_orig=5
  chunk 8: 250,000 linhas | via_geo=249,996 | so_orig=4
  chunk 9: 250,000 linhas | via_geo=249,998 | so_orig=2
  chunk 10: 250,000 linhas | via_geo=249,999 | so_orig=1
  chunk 11: 250,000 linhas | via_geo=250,000 | so_orig=0
  chunk 12: 250,000 linhas | via_geo=250,000 | so_orig=0
  chunk 13: 250,000 linhas | via_geo=249,998 | so_orig=2
  chunk 14: 250,000 linhas | via_geo=249,995 | so_orig=5
  chunk 15: 250,000 linhas | via_geo=249,999 | so_orig=1
  chunk 16: 250,0

## Etapa 3 — Agregar setor → CEPs (cascata híbrida)

Para cada setor: agrega CEPs do `setor_cep_geo`. Setores ausentes em `setor_cep_geo` mas presentes em `setor_cep_orig` ganham fallback (`origem_cep = cnefe_original`).

In [9]:
AGG_SQL = '''
    SELECT cd_setor,
           COUNT(*) AS qtd_ceps,
           MIN(cep) AS cep_inicial,
           MAX(cep) AS cep_final,
           CASE WHEN MIN(cep) = MAX(cep) THEN MIN(cep)
                ELSE MIN(cep) || ' - ' || MAX(cep) END AS faixa_cep,
           SUM(qtd_enderecos) AS total_enderecos,
           GROUP_CONCAT(cep, '|') AS lista_ceps
    FROM {tabela}
    GROUP BY cd_setor
'''

with open_sqlite(WORK_SQLITE) as conn:
    agg_geo  = pd.read_sql_query(AGG_SQL.format(tabela='setor_cep_geo'),  conn)
    agg_orig = pd.read_sql_query(AGG_SQL.format(tabela='setor_cep_orig'), conn)

log(f'agg_geo  : {len(agg_geo):,}')
log(f'agg_orig : {len(agg_orig):,}')

# Filtrar so UF de interesse (codigos com prefixo correto).
agg_geo  = agg_geo.loc[agg_geo['cd_setor'].str.slice(0, 2) == UF_CODIGO].copy()
agg_orig = agg_orig.loc[agg_orig['cd_setor'].str.slice(0, 2) == UF_CODIGO].copy()

setores_em_geo = set(agg_geo['cd_setor'])
agg_orig_only  = agg_orig.loc[~agg_orig['cd_setor'].isin(setores_em_geo)].copy()

agg_geo['origem_cep']       = 'geofencing'
agg_orig_only['origem_cep'] = 'cnefe_original'

setor_ceps = pd.concat([agg_geo, agg_orig_only], ignore_index=True)
log(f'Setores agregados (geo + fallback orig): {len(setor_ceps):,}')
print(setor_ceps['origem_cep'].value_counts())
display(setor_ceps.head())

agg_geo  : 102,133
agg_orig : 100,723
Setores agregados (geo + fallback orig): 111,370
origem_cep
geofencing        102133
cnefe_original      9237
Name: count, dtype: int64


,cd_setor,qtd_ceps,cep_inicial,cep_final,faixa_cep,total_enderecos,lista_ceps,origem_cep
0,350010505000001,9,17800009,17800049,17800009 - 17800049,516,17800009|17800011|17800015|17800017|17800041|1...,geofencing
1,350010505000002,11,17800009,17800059,17800009 - 17800059,518,17800009|17800011|17800013|17800017|17800019|1...,geofencing
2,350010505000003,15,17800001,17805048,17800001 - 17805048,535,17800001|17800003|17800005|17800007|17800009|1...,geofencing
3,350010505000004,27,17800003,17805030,17800003 - 17805030,415,17800003|17800005|17800007|17800009|17800037|1...,geofencing
4,350010505000005,23,17800009,17803148,17800009 - 17803148,448,17800009|17800011|17800015|17800017|17800035|1...,geofencing


## Etapa 4 — Atributos do shapefile

Carrega atributos sem geometria, incluindo `CD_BAIRRO`, `CD_DIST`, `CD_SUBDIST` que serão usados na imputação por cascata.

In [10]:
shp_attrs = gpd.read_file(
    SHAPEFILE, ignore_geometry=True,
    columns=['CD_SETOR','CD_UF','NM_UF','CD_MUN','NM_MUN','CD_DIST','NM_DIST',
             'CD_SUBDIST','NM_SUBDIST','CD_BAIRRO','NM_BAIRRO','SITUACAO','CD_TIPO','AREA_KM2'],
)
shp_attrs['cd_setor'] = shp_attrs['CD_SETOR'].astype(str).str.strip()
shp_attrs['cod_uf']   = shp_attrs['CD_UF'].astype(str).str.strip().str.zfill(2)
shp_attrs['cod_mun']  = shp_attrs['CD_MUN'].astype(str).str.strip().str.zfill(7)
shp_attrs['nm_uf']    = shp_attrs['NM_UF'].astype(str).str.strip()
shp_attrs['nm_mun']   = shp_attrs['NM_MUN'].astype(str).str.strip()
shp_attrs['area_km2'] = pd.to_numeric(shp_attrs['AREA_KM2'], errors='coerce')
shp_attrs = shp_attrs.loc[shp_attrs['cod_uf'] == UF_CODIGO]
shp_attrs = shp_attrs.loc[shp_attrs['cd_setor'].str.len() == 15].reset_index(drop=True)
log(f'Setores no shapefile ({UF_SIGLA}): {len(shp_attrs):,}')
display(shp_attrs.head(2))

Setores no shapefile (SP): 103,319


,CD_SETOR,SITUACAO,CD_TIPO,AREA_KM2,CD_UF,NM_UF,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_SUBDIST,NM_SUBDIST,CD_BAIRRO,NM_BAIRRO,cd_setor,cod_uf,cod_mun,nm_uf,nm_mun,area_km2
0,351300905000799,Urbana,0,0.0124,35,São Paulo,3513009,Cotia,351300905,Cotia,35130090500,NaN,NaN,NaN,351300905000799,35,3513009,São Paulo,Cotia,0.0124
1,351510305000201,Urbana,0,0.0134,35,São Paulo,3515103,Embu-Guaçu,351510305,Embu-Guaçu,35151030500,NaN,NaN,NaN,351510305000201,35,3515103,São Paulo,Embu-Guaçu,0.0134


## Etapa 5 — Outer join (driver=renda) e categoria `fora_csv_ibge`

Junta renda (driver) com `setor_ceps` (outer join), mais shapefile attrs. Setores que estão na malha CD2022 e têm CEP via geofencing mas o IBGE não publicou renda recebem `motivo_renda = fora_csv_ibge`.

In [11]:
out = renda_df.merge(setor_ceps, on='cd_setor', how='outer')
veio_so_de_cnefe = out['motivo_renda'].isna() & out['qtd_ceps'].notna()
out.loc[veio_so_de_cnefe, 'motivo_renda'] = 'fora_csv_ibge'

out['tem_cep'] = out['qtd_ceps'].notna().astype(int)
for col in ['qtd_ceps', 'total_enderecos']:
    out[col] = out[col].fillna(0).astype('int64')

out = out.merge(shp_attrs, on='cd_setor', how='left')
out['sigla_uf'] = out['cd_setor'].str.slice(0, 2).map(SIGLA_POR_UF)
out['esta_no_shapefile'] = out['cod_uf'].notna().astype(int)

# Filtra para manter so setores que existem no shapefile CD2022.
n_pre = len(out)
out = out.loc[out['esta_no_shapefile'] == 1].copy()
log(f'Filtro shapefile: {n_pre:,} -> {len(out):,} (removidos {n_pre - len(out):,} codigos antigos do CNEFE)')

log(f'  com renda no CSV (numerica/sigilo/ausente): {int(out["motivo_renda"].isin(["numerica","sigilo","ausente"]).sum()):,}')
log(f'  fora_csv_ibge (sem renda mas com CEP): {int((out["motivo_renda"] == "fora_csv_ibge").sum()):,}')
display(out.head(3))

Filtro shapefile: 111,378 -> 102,599 (removidos 8,779 codigos antigos do CNEFE)
  com renda no CSV (numerica/sigilo/ausente): 100,928
  fora_csv_ibge (sem renda mas com CEP): 1,671


,cd_setor,motivo_renda,renda_v06001,renda_v06002,renda_v06003,renda_v06004,renda_v06005,renda_v06006,qtd_ceps,cep_inicial,cep_final,faixa_cep,total_enderecos,lista_ceps,origem_cep,tem_cep,CD_SETOR,SITUACAO,CD_TIPO,AREA_KM2,CD_UF,NM_UF,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_SUBDIST,NM_SUBDIST,CD_BAIRRO,NM_BAIRRO,cod_uf,cod_mun,nm_uf,nm_mun,area_km2,sigla_uf,esta_no_shapefile
0,350010505000001,numerica,138.0000,288.0000,1.4400,"4,107.2900","24,164,514.2500","2,500.0000",9,17800009,17800049,17800009 - 17800049,516,17800009|17800011|17800015|17800017|17800041|1...,geofencing,1,350010505000001,Urbana,0,0.1238,35,São Paulo,3500105,Adamantina,350010505,Adamantina,35001050500,NaN,NaN,NaN,35,3500105,São Paulo,Adamantina,0.1238,SP,1
1,350010505000002,numerica,280.0000,674.0000,2.1500,"4,446.5500","23,063,356.4800","3,000.0000",11,17800009,17800059,17800009 - 17800059,518,17800009|17800011|17800013|17800017|17800019|1...,geofencing,1,350010505000002,Urbana,0,0.2026,35,São Paulo,3500105,Adamantina,350010505,Adamantina,35001050500,NaN,NaN,NaN,35,3500105,São Paulo,Adamantina,0.2026,SP,1
2,350010505000003,numerica,198.0000,379.0000,1.0400,"4,713.6800","19,642,814.0700","3,050.0000",15,17800001,17805048,17800001 - 17805048,535,17800001|17800003|17800005|17800007|17800009|1...,geofencing,1,350010505000003,Urbana,0,0.2350,35,São Paulo,3500105,Adamantina,350010505,Adamantina,35001050500,NaN,NaN,NaN,35,3500105,São Paulo,Adamantina,0.2350,SP,1


## Etapa 6 — Estatísticas intermediárias (antes da imputação)

In [12]:
stats_inter = pd.DataFrame({
    'metrica': [
        'setores no output total',
        'com CEP (tem_cep=1)',
        'sem CEP (tem_cep=0)',
        'renda numerica',
        'renda sigilo (X)',
        'renda ausente',
        'fora_csv_ibge (com CEP, sem renda)',
        'origem_cep geofencing',
        'origem_cep cnefe_original (fallback)',
    ],
    'valor': [
        len(out),
        int(out['tem_cep'].sum()),
        int((out['tem_cep'] == 0).sum()),
        int((out['motivo_renda'] == 'numerica').sum()),
        int((out['motivo_renda'] == 'sigilo').sum()),
        int((out['motivo_renda'] == 'ausente').sum()),
        int((out['motivo_renda'] == 'fora_csv_ibge').sum()),
        int((out['origem_cep'] == 'geofencing').sum()),
        int((out['origem_cep'] == 'cnefe_original').sum()),
    ],
})
display(stats_inter)

,metrica,valor
0,setores no output total,102599
1,com CEP (tem_cep=1),102591
2,sem CEP (tem_cep=0),8
3,renda numerica,99223
4,renda sigilo (X),1697
5,renda ausente,8
6,"fora_csv_ibge (com CEP, sem renda)",1671
7,origem_cep geofencing,102133
8,origem_cep cnefe_original (fallback),458


## Etapa 7 — Imputação de renda em cascata

Para setores sem renda numérica com `CD_TIPO ∈ {0, 1}` (urbano comum e rural comum), usa a mediana dos vizinhos no nível mais granular que tem `n_amostra ≥ 5`:

1. bairro (`cod_mun + CD_DIST + CD_SUBDIST + CD_BAIRRO`)
2. subdistrito
3. distrito
4. município (fallback)

Setores com `CD_TIPO ∈ {2..9}` (especiais) ficam marcados como `sem_dado_legitimo_tipo_X` — não imputa.

In [13]:
NIVEIS = [
    ('bairro',      ['cod_mun', 'CD_DIST', 'CD_SUBDIST', 'CD_BAIRRO']),
    ('subdistrito', ['cod_mun', 'CD_DIST', 'CD_SUBDIST']),
    ('distrito',    ['cod_mun', 'CD_DIST']),
    ('municipio',   ['cod_mun']),
]


def build_lookups(com_renda_df, niveis):
    out_ = {}
    for nome, keys in niveis:
        agg = (com_renda_df.groupby(keys, dropna=False)
                          .agg(median_v06004=('renda_v06004', 'median'),
                               median_v06006=('renda_v06006', 'median'),
                               n_amostra=('renda_v06004', 'count'))
                          .reset_index())
        out_[nome] = (keys, agg)
    return out_


com_renda = out.loc[out['renda_v06004'].notna()].copy()
log(f'Amostra para lookup (setores com renda numerica): {len(com_renda):,}')
lookups = build_lookups(com_renda, NIVEIS)
for nome, (keys, agg) in lookups.items():
    com_amostra = int((agg['n_amostra'] >= MIN_VIZINHOS).sum())
    log(f'  {nome:11} | grupos: {len(agg):,}  |  com >= {MIN_VIZINHOS} amostras: {com_amostra:,}')

Amostra para lookup (setores com renda numerica): 99,223
  bairro      | grupos: 3,113  |  com >= 5 amostras: 1,944
  subdistrito | grupos: 1,042  |  com >= 5 amostras: 906
  distrito    | grupos: 1,042  |  com >= 5 amostras: 906
  municipio   | grupos: 645  |  com >= 5 amostras: 638


In [14]:
def apply_imputation(df, lookups, tipos_imp, min_n):
    res = df.copy()
    res['renda_v06004_estimada'] = res['renda_v06004']
    res['renda_v06006_estimada'] = res['renda_v06006']
    res['origem_renda'] = 'original'

    sem_renda_mask = res['renda_v06004'].isna()
    res.loc[sem_renda_mask, 'origem_renda'] = pd.NA

    tipo_str = res['CD_TIPO'].astype(str)
    nao_imp_mask = sem_renda_mask & ~tipo_str.isin(tipos_imp)
    res.loc[nao_imp_mask, 'origem_renda'] = 'sem_dado_legitimo_tipo_' + tipo_str[nao_imp_mask]

    needs_imp = sem_renda_mask & tipo_str.isin(tipos_imp)
    log(f'Candidatos a imputacao (sem renda + CD_TIPO in {sorted(tipos_imp)}): {int(needs_imp.sum()):,}')

    for nome, (keys, agg) in lookups.items():
        if not needs_imp.any():
            break
        agg_valid = agg.loc[agg['n_amostra'] >= min_n, keys + ['median_v06004', 'median_v06006']]
        if agg_valid.empty:
            continue
        candidatos = res.loc[needs_imp].reset_index().rename(columns={'index': '_orig_idx'})
        merged = candidatos.merge(agg_valid, on=keys, how='left')
        ok = merged['median_v06004'].notna()
        if ok.any():
            orig_idx = merged.loc[ok, '_orig_idx'].values
            res.loc[orig_idx, 'renda_v06004_estimada'] = merged.loc[ok, 'median_v06004'].values
            res.loc[orig_idx, 'renda_v06006_estimada'] = merged.loc[ok, 'median_v06006'].values
            res.loc[orig_idx, 'origem_renda'] = f'imputado_{nome}'
            needs_imp.loc[orig_idx] = False
            log(f'  nivel {nome:11}: {int(ok.sum()):,} setores imputados')

    res.loc[needs_imp, 'origem_renda'] = 'imputacao_sem_amostra'
    return res


out_final = apply_imputation(out, lookups, TIPOS_IMPUTAVEIS, MIN_VIZINHOS)
log(f'Linhas no output final: {len(out_final):,}')

Candidatos a imputacao (sem renda + CD_TIPO in ['0', '1']): 1,053
  nivel bairro     : 927 setores imputados
  nivel subdistrito: 114 setores imputados
  nivel municipio  : 11 setores imputados
Linhas no output final: 102,599


## Etapa 8 — Estatísticas finais e exportar

In [15]:
print('Distribuicao de origem_renda:')
origem_counts = out_final['origem_renda'].value_counts(dropna=False)
origem_pct = (origem_counts / len(out_final) * 100).round(2)
display(pd.DataFrame({'setores': origem_counts, 'pct': origem_pct}))

n_total       = len(out_final)
n_original    = int((out_final['origem_renda'] == 'original').sum())
n_imputados   = int(out_final['origem_renda'].astype(str).str.startswith('imputado_').sum())
n_legitimo    = int(out_final['origem_renda'].astype(str).str.startswith('sem_dado_legitimo_').sum())
n_renda_final = int(out_final['renda_v06004_estimada'].notna().sum())
n_com_cep     = int((out_final['tem_cep'] == 1).sum())

print()
print(f'Total setores              : {n_total:,}')
print(f'  com renda original       : {n_original:,} ({n_original/n_total*100:.2f}%)')
print(f'  com renda imputada       : {n_imputados:,} ({n_imputados/n_total*100:.2f}%)')
print(f'  legitimos sem renda      : {n_legitimo:,} ({n_legitimo/n_total*100:.2f}%)')
print()
print(f'COBERTURA RENDA (origin+imp): {n_renda_final:,} / {n_total:,} = {n_renda_final/n_total*100:.2f}%')
print(f'COBERTURA CEP              : {n_com_cep:,} / {n_total:,} = {n_com_cep/n_total*100:.2f}%')

Distribuicao de origem_renda:


,setores,pct
origem_renda,,
original,99223,96.7100
sem_dado_legitimo_tipo_4,1948,1.9000
imputado_bairro,927,0.9000
sem_dado_legitimo_tipo_7,196,0.1900
sem_dado_legitimo_tipo_6,148,0.1400
imputado_subdistrito,114,0.1100
sem_dado_legitimo_tipo_3,14,0.0100
imputado_municipio,11,0.0100
sem_dado_legitimo_tipo_2,7,0.0100



Total setores              : 102,599
  com renda original       : 99,223 (96.71%)
  com renda imputada       : 1,052 (1.03%)
  legitimos sem renda      : 2,323 (2.26%)

COBERTURA RENDA (origin+imp): 100,275 / 102,599 = 97.73%
COBERTURA CEP              : 102,591 / 102,599 = 99.99%


In [16]:
# Ordem final das colunas.
final_columns = [
    'cd_setor', 'sigla_uf', 'cod_uf', 'nm_uf',
    'cod_mun', 'nm_mun', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST',
    'CD_BAIRRO', 'NM_BAIRRO', 'SITUACAO', 'CD_TIPO', 'area_km2',
    'motivo_renda', 'origem_renda',
    'renda_v06001', 'renda_v06002', 'renda_v06003',
    'renda_v06004', 'renda_v06005', 'renda_v06006',
    'renda_v06004_estimada', 'renda_v06006_estimada',
    'origem_cep', 'tem_cep', 'qtd_ceps',
    'cep_inicial', 'cep_final', 'faixa_cep',
    'total_enderecos', 'lista_ceps', 'esta_no_shapefile',
]
ordem = [c for c in final_columns if c in out_final.columns]
out_ordenado = out_final[ordem].copy()

out_ordenado.to_parquet(OUT_PARQUET, index=False)
log(f'[export] Parquet: {OUT_PARQUET}')
out_ordenado.to_csv(OUT_CSV, sep=';', index=False, encoding='utf-8')
log(f'[export] CSV: {OUT_CSV}')

resumo = {
    'generated_at': now_iso(),
    'uf_filter': UF_SIGLA,
    'tipos_imputaveis': sorted(TIPOS_IMPUTAVEIS),
    'min_vizinhos': MIN_VIZINHOS,
    'setores_total': n_total,
    'setores_com_renda_original': n_original,
    'setores_imputados': n_imputados,
    'setores_legitimos_sem_renda': n_legitimo,
    'cobertura_renda_pct': round(n_renda_final / n_total * 100, 4),
    'cobertura_cep_pct':   round(n_com_cep / n_total * 100, 4),
    'distribuicao_origem_renda': origem_counts.to_dict(),
    'cnefe_stats': cnefe_stats,
    'outputs': {
        'parquet': str(OUT_PARQUET),
        'csv':     str(OUT_CSV),
        'sqlite':  str(WORK_SQLITE),
        'summary': str(OUT_RESUMO),
    },
}
OUT_RESUMO.write_text(json.dumps(resumo, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
log(f'[export] Resumo: {OUT_RESUMO}')
print()
print(json.dumps(resumo, ensure_ascii=False, indent=2, default=str)[:1500])

[export] Parquet: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp\renda_setor_cep_sp_final.parquet
[export] CSV: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp\renda_setor_cep_sp_final.csv
[export] Resumo: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp\renda_setor_cep_sp_final_resumo.json

{
  "generated_at": "2026-05-25T23:06:23+00:00",
  "uf_filter": "SP",
  "tipos_imputaveis": [
    "0",
    "1"
  ],
  "min_vizinhos": 5,
  "setores_total": 102599,
  "setores_com_renda_original": 99223,
  "setores_imputados": 1052,
  "setores_legitimos_sem_renda": 2323,
  "cobertura_renda_pct": 97.7349,
  "cobertura_cep_pct": 99.9922,
  "distribuicao_origem_renda": {
    "original": 99223,
    "sem_dado_legitimo_tipo_4": 1948,
    "imputado_bairro": 927,
    "sem_dado_legitimo_tipo_7": 196,
    "sem_dado_legitimo_tipo_6": 148,
    "imputado_subdistrito": 114,
    "sem_dado_legitimo_tipo_3": 14,
    "imputado_municipio": 11,
    "sem_dado_legitimo_tipo_2"

## Notas finais

- **Output principal**: `saida_etl_final_sp/renda_setor_cep_sp_final.parquet` (1 linha = 1 setor da CD2022 SP).
- **Cobertura CEP esperada**: ~99% dos setores ganham CEP via geofencing.
- **Cobertura renda esperada**: ~97,25% via original + imputação cascateada.
- **Os ~2,75% sem renda** são `sem_dado_legitimo_tipo_X` — setores especiais (parques, áreas industriais, favelas com sigilo IBGE). Para esses, imputar não faz sentido conceitual.
- **Próximos passos**: rodar `2_analises_diagnostico.ipynb` para analisar os resultados e investigar se vale tentar chegar a 100% via KNN espacial ou modelagem.